In [5]:
import pandas as pd
import numpy as np
import os
from sklearn.preprocessing import StandardScaler


In [6]:
# One hot encode and normalize
def preprocess_data(traindata, testdata):
    y_train = traindata['gname']
    y_test = testdata['gname']

    train_features = traindata.drop(columns=['gname'])
    test_features = testdata.drop(columns=['gname'])

    geodata = ['longitude', 'latitude']
    numeric_cols = [col for col in train_features.columns if col not in geodata]

    combined_geo = pd.concat([train_features[geodata], test_features[geodata]])
    geo_onehot = pd.get_dummies(combined_geo, columns=geodata)

    train_geo = geo_onehot.iloc[:len(train_features)]
    test_geo = geo_onehot.iloc[len(train_features):]

    scaler = StandardScaler()
    train_num = pd.DataFrame(scaler.fit_transform(train_features[numeric_cols]), columns=numeric_cols, index=train_features.index)
    test_num = pd.DataFrame(scaler.transform(test_features[numeric_cols]), columns=numeric_cols, index=test_features.index)

    X_train = pd.concat([train_num, train_geo], axis=1)
    X_test = pd.concat([test_num, test_geo], axis=1)

    traindata = pd.concat([X_train, y_train], axis=1)
    testdata = pd.concat([X_test, y_test], axis=1)

    return traindata, testdata


In [7]:
if not os.path.isdir("train"):
    os.mkdir("train")
if not os.path.isdir("test"):
    os.mkdir("test")

In [8]:
trainpath = '../traindata'
testpath = '../testdata'

partitions = [100, 200, 300, 478]

for partition in partitions:
    traindata = pd.read_csv(f'{trainpath}/train{partition}.csv', encoding='ISO-8859-1')
    testdata = pd.read_csv(f'{testpath}/test{partition}.csv', encoding='ISO-8859-1')

    # Drop irrelevant columns
    cols_to_drop = ['Unnamed: 0', 'country', 'city', 'region', 'provstate', 'natlty1', 'specificity', 'iyear', 'imonth', 'iday']
    traindata = traindata.drop(columns=cols_to_drop)
    testdata = testdata.drop(columns=cols_to_drop)

    # Preprocess features
    traindata, testdata = preprocess_data(traindata, testdata)

    traindata.to_csv(f'train/train{partition}.csv', index=False)
    testdata.to_csv(f'test/test{partition}.csv', index=False)
